# Shipping emissions calculator (Bare Necessities demo)

Distance-based method (GHG Protocol / GLEC): weight × distance × emission factor for each leg.

**Demo orders are synthetic.** Emission factors and their sources are in `data/emission_factors.csv`.

Run the cells from top to bottom (Shift + Enter). This notebook must sit in the project folder, next to `emissions.py` and the `data` folder.

In [ ]:
# Install the packages once (skip if already installed)
%pip install -q pandas numpy matplotlib

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from generate_orders import generate
from emissions import calculate, load_factors, summarise

## 1. Emission factors and assumptions

In [ ]:
factors = load_factors()
factors[["low", "central", "high", "unit", "status"]]

## 2. Orders

Synthetic demo orders. To use a real export instead, replace this cell with `orders = pd.read_csv("your_file.csv")` (columns: order_id, city, channel, product_weight_kg, mode).

In [ ]:
orders = generate(5000)
orders.to_csv("data/synthetic_orders.csv", index=False)
print(len(orders), "orders;", f"{(orders['mode'] == 'air').mean():.0%} by air")
orders.head()

## 3. Results: low / central / high

In [ ]:
summary = summarise(orders, factors)
summary.round(3)

In [ ]:
res = calculate(orders, factors, "central")
res.groupby(["channel", "mode"])["kg_co2e_total"].agg(orders="count", total_kg="sum", avg_kg="mean").round(3)

## 4. Emissions by destination city (central case)

In [ ]:
by_city = res.groupby("city")["kg_co2e_total"].sum().sort_values()
ax = by_city.plot.barh(figsize=(7, 6), color="#4a7c59")
ax.set_xlabel("kg CO2e (central case)")
ax.set_ylabel("")
ax.set_title("Delivery emissions by destination city")
plt.tight_layout()
plt.show()

## 5. Same parcel, road vs air

A 450 g parcel from Bengaluru to Delhi.

In [ ]:
example = pd.DataFrame([{"order_id": m, "city": "Delhi", "channel": "website", "product_weight_kg": 0.45, "mode": m} for m in ["surface", "air"]])
cmp = calculate(example, factors)[["mode", "road_km", "air_km", "kg_co2e_total"]]
cmp["g_co2e"] = (cmp["kg_co2e_total"] * 1000).round(0)
print(f"Air emits {cmp.g_co2e.iloc[1] / cmp.g_co2e.iloc[0]:.0f}x more than road for this parcel")
cmp

## 6. Save per-order results

In [ ]:
res.to_csv("emissions_results.csv", index=False)
print("Saved emissions_results.csv")